# RadLE Meta Muse Spark Colab Pro

Hosted Meta Model API runner for `muse-spark-1.1`. This notebook pulls the current GitHub code first, imports the RadLE Python helpers from `src/`, reads the Meta API key from Colab Secrets, and runs a smoke before any full benchmark.

In [ ]:
# ==========================================
# 1. COLAB SETUP: REPO CODE ONLY
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_REF = os.environ.get("RADLE_REPO_REF", "codex/meta-muse-spark-colab")
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_meta_model_api_runtime.py"

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None


def authenticated_repo_url():
    if not github_token:
        return REPO_URL
    return REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")


def mask_secret(value):
    text = str(value)
    if github_token:
        text = text.replace(github_token, "***")
    return text


def run_command(args, check=True, env=None):
    result = subprocess.run(args, text=True, capture_output=True, env=env)
    stdout = mask_secret(result.stdout)
    stderr = mask_secret(result.stderr)
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        safe_args = [mask_secret(arg) for arg in args[:4]]
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {safe_args}")
    return result


repo_url = authenticated_repo_url()
if (REPO_DIR / ".git").exists():
    if github_token:
        run_command(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url])
    run_command(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF])
    checkout_result = run_command(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=False)
    if checkout_result.returncode != 0:
        run_command(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"])
    pull_result = run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=False)
    if pull_result.returncode != 0:
        raise RuntimeError(
            "git pull failed; the Colab checkout may be stale or locally modified. "
            f"Restart the runtime or remove {REPO_DIR}, then rerun this setup cell."
        )
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )
    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone the RadLE source module."
        )
    run_command(["git", "clone", "--branch", REPO_REF, repo_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Meta runtime module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

branch_result = run_command(["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"], check=False)
commit_result = run_command(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=False)
if branch_result.returncode == 0 and commit_result.returncode == 0:
    print("Using repo checkout:", branch_result.stdout.strip(), commit_result.stdout.strip())


In [ ]:
# ==========================================
# 2. DEPENDENCIES + RADLE IMPORTS
# ==========================================
import importlib
import inspect

base_packages = [
    "openai",
    "pandas",
    "anthropic",
    "google-genai",
]
run_command([sys.executable, "-m", "pip", "install", "-q", *base_packages])

import radle_meta_model_api_runtime as meta_runtime
import radle_benchmark

radle_benchmark = importlib.reload(radle_benchmark)
meta_runtime = importlib.reload(meta_runtime)
meta_runtime.radle_benchmark = radle_benchmark
meta_runtime.configure_benchmark_runtime()

for _required_param in ("backup_dir", "models", "resume"):
    if _required_param not in inspect.signature(radle_benchmark.run_benchmark).parameters:
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_param}. "
            "Rerun setup after git pull, or restart the runtime."
        )

for _required_function in (
    "audit_benchmark_output",
    "run_targeted_repair",
    "build_run_paths",
    "promote_final_results",
    "export_public_release_tables",
    "create_scorer_view",
):
    if not hasattr(radle_benchmark, _required_function):
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_function}. "
            "Rerun setup after git pull, or restart the runtime."
        )

print("Benchmark module:", radle_benchmark.__file__)
print("Meta runtime module:", meta_runtime.__file__)
meta_runtime.print_model_roster()


In [ ]:
# ==========================================
# 3. META MODEL API SECRET + TEXT PROBE
# ==========================================
MODEL_API_BASE_URL = os.environ.get("META_MODEL_API_BASE_URL", meta_runtime.DEFAULT_BASE_URL)
MODEL_API_KEY = meta_runtime.get_secret("MODEL_API_KEY", "META_MODEL_API_KEY")
if not MODEL_API_KEY:
    raise RuntimeError(
        "Missing Meta Model API key. Add a Colab secret named MODEL_API_KEY "
        "or META_MODEL_API_KEY. Do not paste the key into this notebook."
    )

meta_client = meta_runtime.make_openai_client(
    api_key=MODEL_API_KEY,
    base_url=MODEL_API_BASE_URL,
)

RUN_TEXT_PROBE = True
print("Meta API base URL:", MODEL_API_BASE_URL)
print("Selected model:", meta_runtime.MODEL_ID)
if RUN_TEXT_PROBE:
    probe_text = meta_runtime.text_probe(meta_client)
    print("Text probe response:", probe_text)


In [ ]:
# ==========================================
# 4. DRIVE DATASET + RUN CONFIG
# ==========================================
from pathlib import Path

# Start with a 1-case smoke. Set TEST_LIMIT=None only for the collaborator's full run.
TEST_LIMIT = 1
RUN_LABEL = "meta_muse_spark_1case"
RESUME = True
MAX_OUTPUT_TOKENS = meta_runtime.META_MAX_OUTPUT_TOKENS
UNIVERSAL_TEMPERATURE = radle_benchmark.UNIVERSAL_TEMPERATURE

DEFAULT_DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset"
)
DATASET_ROOT_OVERRIDE = os.environ.get("RADLE_DATASET_ROOT", "")


def has_master_data(path):
    return (Path(path) / "RadLE v2 Master Data").exists()


if DATASET_ROOT_OVERRIDE:
    dataset_root = Path(DATASET_ROOT_OVERRIDE)
else:
    if not has_master_data(DEFAULT_DRIVE_DATASET_ROOT):
        from google.colab import drive
        print("Mounting Google Drive for the default RadLE dataset path...")
        drive.mount("/content/drive")
    dataset_root = DEFAULT_DRIVE_DATASET_ROOT

if not has_master_data(dataset_root):
    raise RuntimeError(
        "Could not find a folder named 'RadLE v2 Master Data' under dataset_root. "
        f"Checked: {dataset_root}. Set RADLE_DATASET_ROOT to the RadLE dataset root if needed."
    )

run_paths = meta_runtime.build_meta_run_paths(
    dataset_root,
    run_label=RUN_LABEL,
)

master_images_folder = run_paths["master_images_folder"]
raw_results_csv = run_paths["raw_results_csv"]
raw_backup_dir = run_paths["raw_backup_dir"]
scorer_csv = run_paths["scorer_view_csv"]
repair_output_csv = run_paths["repair_results_csv"]
repair_call_log_csv = run_paths["repair_call_log_csv"]
repair_plan_csv = run_paths["repair_plan_csv"]
repair_backup_dir = run_paths["repair_backup_dir"]
final_results_csv = run_paths["final_results_csv"]
final_manifest_json = run_paths["final_manifest_json"]
public_release_dir = run_paths["public_release_dir"]

active_model = [meta_runtime.get_model_config()]
if active_model[0]["name"] != "muse_spark_1_1" or active_model[0]["id"] != "muse-spark-1.1":
    raise RuntimeError(f"Wrong active model config: {active_model}")

print("Dataset root:", dataset_root)
print("Master images:", master_images_folder)
print("Run folder:", run_paths["run_root"])
print("Raw results CSV:", raw_results_csv)
print("Existing rows:", meta_runtime.count_existing_rows(raw_results_csv))
print("TEST_LIMIT:", TEST_LIMIT)
print("Active model:", active_model)


In [ ]:
# ==========================================
# 5. RUN ONE-MODEL MUSE SPARK SMOKE / FULL RUN
# ==========================================
from IPython.display import display

df_final = meta_runtime.run_meta_model_benchmark(
    client=meta_client,
    image_folder=master_images_folder,
    output_csv=raw_results_csv,
    test_limit=TEST_LIMIT,
    backup_dir=raw_backup_dir,
    resume=RESUME,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    universal_temperature=UNIVERSAL_TEMPERATURE,
)

print("Raw results CSV:", raw_results_csv)
display(df_final.tail(min(5, len(df_final))))


In [ ]:
# ==========================================
# 6. SCORER VIEW + READ-ONLY AUDIT
# ==========================================
df_scorer, display_df, scorer_csv = radle_benchmark.create_scorer_view(
    raw_results_csv,
    scorer_csv=scorer_csv,
)
print("Scorer CSV:", scorer_csv)
display(display_df)

audit_expected_case_ids = None
if TEST_LIMIT is not None:
    audit_expected_case_ids = set(df_final["Master_Case_ID"].astype(str).tolist())

audit = radle_benchmark.audit_benchmark_output(
    raw_csv=raw_results_csv,
    models=active_model,
    expected_case_ids=audit_expected_case_ids,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
audit_table = audit["audit"]
repair_targets = audit["repair_targets"]
no_paid_cleanup = audit["no_paid_cleanup"]

print("Rows:", len(audit_table))
print("Repair targets:", len(repair_targets))
print("No-paid cleanup targets:", len(no_paid_cleanup))
display(audit["dataset_integrity"])
display(audit["bucket_summary"])
display(repair_targets.head(50))


In [ ]:
# ==========================================
# 7. TARGETED REPAIR PLAN / RUN
# ==========================================
# Keep REPAIR_CONFIRMATION="NO" to preview without API calls or writes.
# Use "YES_REPAIR_10" for a capped paid repair, then "YES_REPAIR_ALL" if needed.
REPAIR_CONFIRMATION = "NO"

repair_results = radle_benchmark.run_targeted_repair(
    client=meta_client,
    image_folder=master_images_folder,
    input_csv=raw_results_csv,
    output_csv=repair_output_csv,
    repair_call_log_csv=repair_call_log_csv,
    repair_plan_csv=repair_plan_csv,
    confirmation=REPAIR_CONFIRMATION,
    models=active_model,
    backup_dir=repair_backup_dir,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    universal_temperature=UNIVERSAL_TEMPERATURE,
)

print("API calls this repair run:", repair_results["api_calls_this_run"])
print("Repair output CSV:", repair_results["output_csv"])
display(repair_results["repair_plan"].head(50))


In [ ]:
# ==========================================
# 8. PROMOTE PRIVATE FINAL FILE
# ==========================================
if TEST_LIMIT is not None:
    raise RuntimeError(
        "Do not promote partial smoke output. Set TEST_LIMIT=None after the full run, "
        "then rerun benchmark/audit/repair before promotion."
    )

from pathlib import Path

repair_confirmed = globals().get("REPAIR_CONFIRMATION", "NO") != "NO"
repair_output_ready = repair_confirmed and Path(repair_output_csv).exists()
private_final_source_csv = repair_output_csv if repair_output_ready else raw_results_csv
private_final_source_label = "repaired" if repair_output_ready else "raw"

ALLOW_PROMOTE_WITH_PENDING_REPAIRS = False
audit_pre_promote = radle_benchmark.audit_benchmark_output(
    raw_csv=private_final_source_csv,
    models=active_model,
    expected_case_ids=None,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
pending_repairs = len(audit_pre_promote["repair_targets"])
if pending_repairs > 0 and not ALLOW_PROMOTE_WITH_PENDING_REPAIRS:
    raise RuntimeError(
        f"{pending_repairs} case-model cells still need repair. "
        "Run repair or set ALLOW_PROMOTE_WITH_PENDING_REPAIRS=True intentionally."
    )

final_manifest = radle_benchmark.promote_final_results(
    source_csv=private_final_source_csv,
    final_csv=final_results_csv,
    manifest_json=final_manifest_json,
    run_id=run_paths["run_id"],
    source_label=private_final_source_label,
    metadata={
        "run_label": RUN_LABEL,
        "model_name": meta_runtime.MODEL_NAME,
        "model_id": meta_runtime.MODEL_ID,
        "provider": meta_runtime.META_PROVIDER_LABEL,
        "test_limit": TEST_LIMIT if TEST_LIMIT is not None else "full",
    },
)

print("Private final CSV:", final_results_csv)
print("Private final manifest:", final_manifest_json)
print("Private final SHA256:", final_manifest["sha256"])


In [ ]:
# ==========================================
# 9. EXPORT ANSWER-FREE PUBLIC RELEASE TABLES
# ==========================================
if TEST_LIMIT is not None:
    raise RuntimeError(
        "Do not export public release tables from partial smoke output. "
        "Set TEST_LIMIT=None after the full run and final promotion."
    )

from pathlib import Path

if not Path(final_results_csv).exists():
    raise RuntimeError("final_results_csv does not exist. Promotion was likely halted.")

public_release_files = radle_benchmark.export_public_release_tables(
    results_csv=final_results_csv,
    output_dir=public_release_dir,
    models=active_model,
    call_log_csv=repair_call_log_csv if Path(repair_call_log_csv).exists() else None,
    run_id=run_paths["run_id"],
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

print("Public case-model CSV:", public_release_files["case_model_csv"])
print("Public model summary CSV:", public_release_files["summary_csv"])
print("Public sanitized call log CSV:", public_release_files["sanitized_call_log_csv"])
print("Public manifest:", public_release_files["manifest_json"])
